<a href="https://colab.research.google.com/github/carlosaupta-blip/Esp32_Yolo26_/blob/Entrenamiento_modelo/quantizar_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalar dependencias


In [1]:

%pip install -q numpy<2.0.0
%pip install -q esp-ppq==1.3.11
%pip install -q onnx==1.17.0
%pip install -q onnxruntime>=1.19.0
%pip install -q torch>=2.4.0
%pip install -q torchvision>=0.19.0
%pip install -q ultralytics==8.4.7
%pip install -q onnxsim>=0.4.36
%pip install -q roboflow
# cDnPLZTvGOfV

/bin/bash: line 1: 2.0.0: No such file or directory


🧱 Step 1: Download & Prepare LEGO Minifigures Dataset

In [2]:
# NuJ6TQkbHIt4
from roboflow import Roboflow
from google.colab import drive
drive.mount('/content/drive')
import os
from dotenv import load_dotenv

# Specify the path to your .env file in Google Drive
dotenv_path = '/content/drive/MyDrive/Colab Notebooks/.env'

# Load environment variables from the .env file
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path) # This loads variables into os.environ
    print(f"Loaded .env file from {dotenv_path}")
    # Retrieve the API key from the environment variables
    roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")
else:
    print(f"Warning: .env file not found at {dotenv_path}. Please create one with your ROBOFLOW_API_KEY.")
    # Fallback if .env not found, or if API key isn't in .env
    roboflow_api_key = os.getenv("ROBOFLOW_API_KEY", "YOUR_API_KEY_HERE")

# Ensure roboflow_api_key is a string, even if .env was missing or didn't contain it
if not isinstance(roboflow_api_key, str) or roboflow_api_key == "YOUR_API_KEY_HERE":
    print("Error: Roboflow API key not found or is default. Please set ROBOFLOW_API_KEY in your .env file or hardcode it.")
    # You might want to raise an exception or exit here if the key is critical
    # For now, we'll continue with the placeholder, which will likely fail the Roboflow call


rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("carlos-aponte-upta").project("face-recognation-ismus-fomib")
version = project.version(1)
dataset = version.download("yolo26")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded .env file from /content/drive/MyDrive/Colab Notebooks/.env
loading Roboflow workspace...
loading Roboflow project...


Step 2: Fix Absolute Paths in data.yaml

In [3]:
# S8rWG2xlHNxo
import yaml
import os

# dataset.location comes from the previous cell (NuJ6TQkbHIt4)
yaml_path = os.path.join(dataset.location, 'data.yaml')

# 1. Read the current yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# 2. Fix the path logic
data['path'] = dataset.location # Use the actual downloaded path
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

# 3. Write it back
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml paths updated!")
print(f"New 'path' is: {data['path']}")
DRIVE_SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO26'

✅ data.yaml paths updated!
New 'path' is: /content/Face-Recognation-1


Step 3: Fine-Tuning YOLO26n

In [ ]:
# AqV2d9C0HT2H
from ultralytics import YOLO
import os
import torch

# Disable unwanted loggers
os.environ["COMET_MODE"] = "disabled"
os.environ["WANDB_MODE"] = "disabled"


# Determine the device to use
device = 0 if torch.cuda.is_available() else 'cpu' # Use GPU 0 if available, else CPU
print(f"Using device: {device}")

# Load pre-trained base model
model = YOLO('yolo26n.pt')

# Fine-tune on LEGO dataset
results = model.train(
    data=yaml_path,
    epochs=15,
    imgsz=512,
    batch=24,
    device=device,
    project=os.path.dirname(DRIVE_SAVE_PATH), # Carpeta padre
    name=os.path.basename(DRIVE_SAVE_PATH),    # Nombre de la carpeta del experimento
    plots=False, # Changed from True to False to bypass scipy.ndimage import that causes ImportError

    # MuSGD Fine-tuning setup
    optimizer='MuSGD',
    lr0=0.005,
    lrf=0.01,
)

print("✅ Training complete!")

In [4]:
# hbPgMP__HdR3
import shutil
from glob import glob

# Get the most recently modified best.pt weights trace
weight_files = glob("/content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO26/weights/best.pt")

if weight_files:
    latest_weights = max(weight_files, key=os.path.getctime)
    destination = "yolo26n_face.pt"
    shutil.copy(latest_weights, destination)
    print(f"✅ Success! Extracted Custom weights as: {os.path.abspath(destination)}")
else:
    print("❌ Error: Could not find 'best.pt' in runs/detect/. Did training finish?")

✅ Success! Extracted Custom weights as: /content/yolo26n_face.pt


🛠️ Step 4: YOLOv26 Output Quantization Pipeline

In [5]:
import os
import sys

# =========================================================================
# IMPORTACIÓN DESDE GIT DEL PROYECTO ESP-DL (YOLO26 SCRIPTS Y ESP_PPQ_LUT)
# =========================================================================
# Crear la carpeta 'scripts' si no existe
os.makedirs('scripts', exist_ok=True)

# Definir las URLs base para cada conjunto de archivos
scripts_base_url = "https://raw.githubusercontent.com/espressif/esp-dl/master/examples/tutorial/how_to_quantize_model/quantize_yolo26/scripts/"
esp_ppq_lut_base_url = "https://raw.githubusercontent.com/espressif/esp-dl/master/examples/tutorial/how_to_quantize_model/quantize_yolo26/esp_ppq_lut/"

# Lista de todos los archivos y sus rutas relativas locales, junto con la URL base de donde provienen
required_files_info = [
    # Archivos de la carpeta 'scripts'
    {"local_path": "utils.py", "base_url": scripts_base_url},
    {"local_path": "dataset.py", "base_url": scripts_base_url},
    {"local_path": "esp_ppq_patch.py", "base_url": scripts_base_url},
    {"local_path": "esp_ppq_patch_2.py", "base_url": scripts_base_url},
    {"local_path": "notebook_helpers.py", "base_url": scripts_base_url},
    {"local_path": "export.py", "base_url": scripts_base_url},
    {"local_path": "trainer.py", "base_url": scripts_base_url},
    {"local_path": "validator.py", "base_url": scripts_base_url},
    {"local_path": "__init__.py", "base_url": scripts_base_url},

    # Archivos de la carpeta 'esp_ppq_lut'
    {"local_path": "esp_ppq_lut/__init__.py", "base_url": esp_ppq_lut_base_url},
    {"local_path": "esp_ppq_lut/emulator.py", "base_url": esp_ppq_lut_base_url},
    {"local_path": "esp_ppq_lut/exporter.py", "base_url": esp_ppq_lut_base_url},
    {"local_path": "esp_ppq_lut/passes.py", "base_url": esp_ppq_lut_base_url},
    {"local_path": "esp_ppq_lut/patches.py", "base_url": esp_ppq_lut_base_url},
    {"local_path": "esp_ppq_lut/utils.py", "base_url": esp_ppq_lut_base_url},
    {"local_path": "esp_ppq_lut/verifier.py", "base_url": esp_ppq_lut_base_url},
]

print("📥 Descargando scripts de soporte desde el repositorio de Espressif (raw.githubusercontent.com)...")
for file_info in required_files_info:
    relative_path = file_info["local_path"]
    source_base_url = file_info["base_url"]

    target_local_path = os.path.join('scripts', relative_path)

    # Crear los directorios padres si no existen (ej. 'scripts/esp_ppq_lut/')
    os.makedirs(os.path.dirname(target_local_path), exist_ok=True)

    if not os.path.exists(target_local_path):
        url = source_base_url + relative_path.split('/')[-1] # Obtener solo el nombre del archivo del relative_path
        print(f"  Descargando {relative_path}...")
        # Usar wget para descargar el archivo
        # -q: modo silencioso
        # --show-progress: mostrar barra de progreso
        # -O: guardar el archivo con un nombre específico
        !wget -q --show-progress -O "{target_local_path}" "{url}"
        if os.path.exists(target_local_path):
            print(f"  ✅ Descargado: {relative_path}")
        else:
            print(f"  ❌ Fallo al descargar: {relative_path}")
    else:
        print(f"  ✔️ Ya existe localmente: {relative_path}")

# Asegurar que la ruta 'scripts' tenga prioridad absoluta en sys.path
# Esto es importante para que Python encuentre los módulos locales antes que las librerías del sistema
if os.path.abspath('scripts') not in sys.path:
    sys.path.insert(0, os.path.abspath('scripts'))
# =========================================================================

📥 Descargando scripts de soporte desde el repositorio de Espressif (raw.githubusercontent.com)...
  ✔️ Ya existe localmente: utils.py
  ✔️ Ya existe localmente: dataset.py
  ✔️ Ya existe localmente: esp_ppq_patch.py
  ✔️ Ya existe localmente: esp_ppq_patch_2.py
  ✔️ Ya existe localmente: notebook_helpers.py
  ✔️ Ya existe localmente: export.py
  ✔️ Ya existe localmente: trainer.py
  ✔️ Ya existe localmente: validator.py
  ✔️ Ya existe localmente: __init__.py
  ✔️ Ya existe localmente: esp_ppq_lut/__init__.py
  ✔️ Ya existe localmente: esp_ppq_lut/emulator.py
  ✔️ Ya existe localmente: esp_ppq_lut/exporter.py
  ✔️ Ya existe localmente: esp_ppq_lut/passes.py
  ✔️ Ya existe localmente: esp_ppq_lut/patches.py
  ✔️ Ya existe localmente: esp_ppq_lut/utils.py
  ✔️ Ya existe localmente: esp_ppq_lut/verifier.py


In [6]:
# cICBdkJrHhHH
# ==========================================
# CELL 1: Standard Imports & Paths
# ==========================================
import os
import sys

sys.path.append('scripts')

import torch
import types
from esp_ppq.api import get_target_platform
import esp_ppq.lib as PFL
from esp_ppq.executor import TorchExecutor
from esp_ppq.core import QuantizationVisibility, TargetPlatform
from esp_ppq.api.interface import load_onnx_graph
from esp_ppq.quantization.optim import (
    QuantizeSimplifyPass, QuantizeFusionPass, ParameterQuantizePass,
    RuntimeCalibrationPass, PassiveParameterQuantizePass, QuantAlignmentPass,
    TrainedQuantizationThresholdPass
)


    ___________ ____        ____  ____  ____
   / ____/ ___// __ \      / __ \/ __ \/ __ \
  / __/  \__ \/ /_/ /_____/ /_/ / /_/ / / / /
 / /___ ___/ / ____/_____/ ____/ ____/ /_/ /
/_____//____/_/         /_/   /_/    \___\_\




In [7]:
# NeE-8BgAHl6o
# ==========================================
# CELL 2: User Configurations (LEGO SPECIFIC)
# ==========================================
IMG_SZ_I = 512
PLATFORM = "s3"
PROJECT_NAME = "face"
DATA_YAML_FILE_I = "/content/Face-Recognation-1/data.yaml"
INT16_LUT_STEP_I = 32

In [8]:
# TJk1YRBlHoeH
# ==========================================
# CELL 3: Configuration Injection
# ==========================================
class QATConfig:
    IMG_SZ = IMG_SZ_I
    DEVICE = "cuda" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"
    DATA_YAML_FILE = DATA_YAML_FILE_I
    BATCH_SIZE = 16
    CALIB_MAX_IMAGES = 1200
    CALIB_VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    DATA_FALLBACK_PATH = "/content/drive/MyDrive/Colab Notebooks/Imagenes de calibracion"

    CALIB_STEPS = 16
    QUANT_CALIB_METHOD = "percentile"
    QUANT_ALIGNMENT = "Align to Output"
    TARGET_PLATFORM = get_target_platform("esp32" + PLATFORM, 8)

    INT16_LUT_STEP = INT16_LUT_STEP_I

    # Streaming specific: chunk size for TCN/Streaming inference
    # Typically 16 or 32 depending on ESP32-P4 memory constraints
    STREAMING_CHUNK = None

    # TQT Specific Hyperparameters
    TQT_STEPS = 50
    TQT_LR = 1e-5
    TQT_INT_LAMBDA = 0.25
    TQT_BLOCK_SIZE = 8
    TQT_COLLECTING_DEVICE = DEVICE

    BASE_DIR = os.getcwd()
    MODEL_NAME = f"yolo26n_{PROJECT_NAME}"
    PT_FILE = f"{MODEL_NAME}.pt"
    ONNX_FILE = f"{MODEL_NAME}_export.onnx"

    ESPDL_OUTPUT_DIR = os.path.join(BASE_DIR, "output", f"{PROJECT_NAME}_{IMG_SZ_I}_s8_{PLATFORM}")
    ONNX_PATH = os.path.join(ESPDL_OUTPUT_DIR, ONNX_FILE)

if 'config' not in sys.modules:
    sys.modules['config'] = types.ModuleType('config')
sys.modules['config'].QATConfig = QATConfig

In [9]:
# oD9gAncHHrQH
# ==========================================
# CELL 4: Local Modules & Environment Setup
# ==========================================
from utils import seed_everything, register_mod_op, get_exclusive_ancestors
from dataset import get_calibration_loader
from ultralytics.data.utils import check_det_dataset
from esp_ppq_patch import apply_esp_ppq_patches
from esp_ppq_patch_2 import apply_addlut_patch
from notebook_helpers import extract_model_meta, prepare_onnx, prune_graph_safely
from esp_ppq_lut.passes import EspdlLUTFusionPass
from esp_ppq_lut.exporter import HardwareAwareEspdlExporter

os.makedirs(QATConfig.ESPDL_OUTPUT_DIR, exist_ok=True)

import esp_ppq_lut as esp_lut
esp_lut.initialize(step=QATConfig.INT16_LUT_STEP, verbose=True)

seed_everything(1234)
register_mod_op()
apply_esp_ppq_patches()
apply_addlut_patch()

print("Environment and Configuration setup complete.")

[ESP-PPQ-LUT] Activation forwarders registered for simulation.
[ESPDL Emulator] LUT Operation Handler Registered Globally.
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_INT8
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_INT16
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_S3_INT8
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_S3_INT16
[ESP-PPQ-LUT] Activation forwarders registered for simulation.
[ESP-PPQ-LUT] Extension Initialized (Default Step=32)
Registered 'Mod' handler for PPQ.
Applying ESP-PPQ Runtime Patches...
  [x] Patched OnnxParser.refine_graph
  [x] Patched Backend: Slice
  [x] Patched Backend: Gather
ESP-PPQ Runtime Patches Applied Successfully.
Applying fix to AddLUTPattern.export for correct LUT step propagation...
Environment and Configuration setup complete.


In [10]:
# gIZyQNxdHuCP

# ==========================================
# CELL 5: ONNX Export & Metadata Extraction
# ==========================================
%pip install onnxscript
import sys
import os
# 1\. Definir la carpeta donde se encuentra notebook\_helpers.py
# (Ajusta la ruta si está dentro de una subcarpeta específica)
REPO_DIR = '/content' # O '/content/Esp32\_Yolo26\_' si clonaste el repo
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
# 2\. Importar explícitamente las funciones desde notebook\_helpers.py
try:
  from notebook_helpers import prepare_onnx, extract_model_meta
  print("✅ Funciones 'prepare_onnx' y 'extract_model_meta' cargadas correctamente.")
except ImportError as e:
  print(f"❌ Error al importar desde notebook_helpers: {e}")

prepare_onnx()
model_meta = extract_model_meta()

✅ Funciones 'prepare_onnx' y 'extract_model_meta' cargadas correctamente.
Applying ESP-DL patches for export...
Patched 2 Attention modules.
Patched Detect module: <class 'ultralytics.nn.modules.head.Detect'>
Ultralytics 8.4.7 🚀 Python-3.12.13 torch-2.9.1+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
>> Fuse method blocked! Keeping all heads.
YOLO26n summary (fused): 146 layers, 2,494,694 parameters, 0 gradients, 5.6 GFLOPs

PyTorch: starting from 'yolo26n_face.pt' with input shape (16, 3, 512, 512) BCHW and output shape(s) ((16, 5, 64, 64), (16, 5, 32, 32), (16, 5, 16, 16), (16, 5, 64, 64), (16, 5, 32, 32), (16, 5, 16, 16)) (5.1 MB)

ONNX: starting export with onnx 1.17.0 opset 13...


W0923 03:46:50.517000 18203 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0923 03:46:51.501000 18203 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0923 03:46:51.503000 18203 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ra

ONNX: simplifying with onnxsim v0.7.3...
ONNX: export success ✅ 6.3s, saved as '/content/output/face_512_s8_s3/yolo26n_face_export.onnx' (9.6 MB)

Export complete (11.5s)
Results saved to /content
Predict:         yolo predict task=detect model=/content/output/face_512_s8_s3/yolo26n_face_export.onnx imgsz=512 
Validate:        yolo val task=detect model=/content/output/face_512_s8_s3/yolo26n_face_export.onnx imgsz=512 data=/content/Face-Recognation-1/data.yaml  
Visualize:       https://netron.app
Exported base ONNX to /content/output/face_512_s8_s3/yolo26n_face_export.onnx
Metadata: NC=1, RegMax=1, Stride=[8.0, 16.0, 32.0]


In [11]:
# 5v3wd7HxHxCH

# ==========================================
# CELL 6: Quantizer Initialization & Branch Separation
# ==========================================
print("Loading ONNX Graph into ESP-PPQ...")
graph = load_onnx_graph(onnx_import_file=QATConfig.ONNX_PATH)

output_names = list(graph.outputs.keys())
aux_ops = set()
main_ops = set()

if len(output_names) >= 6:
    aux_outputs = output_names[0:3]
    main_outputs = output_names[3:6]
    aux_ops = get_exclusive_ancestors(graph, aux_outputs, main_outputs)
    main_ops = get_exclusive_ancestors(graph, main_outputs, aux_outputs)

quantizer = PFL.Quantizer(platform=QATConfig.TARGET_PLATFORM, graph=graph)
dispatching_table = PFL.Dispatcher(graph=graph, method="conservative").dispatch(
    quantizer.quant_operation_types
)

for opname, platform in dispatching_table.items():
    if platform == TargetPlatform.UNSPECIFIED:
        dispatching_table[opname] = TargetPlatform(quantizer.target_platform)

for op in aux_ops:
    if op.name in dispatching_table:
        dispatching_table[op.name] = TargetPlatform.FP32

Loading ONNX Graph into ESP-PPQ...


In [12]:
# g--Uz8u4HzuQ
# ==========================================
# CELL 7: Hardware Precision Targets (INT16 / FP32)
# ==========================================
INT16_PLATFORM = get_target_platform("esp32" + PLATFORM, 16)

# Force high-sensitivity exit layers to INT16
int16_layers = {
    # Neck Exits
    "/model.16/cv2/conv/Conv", "/model.16/cv2/conv/Conv/Swish",
    "/model.19/cv2/conv/Conv", "/model.19/cv2/conv/Conv/Swish",
    "/model.22/cv2/conv/Conv", "/model.22/cv2/conv/Conv/Swish",

    # Box Heads
    "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv",
    "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv",
    "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv",

    # Class Heads
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv"
}

for op in graph.operations.values():
    if op.name in dispatching_table and op.name in int16_layers:
        dispatching_table[op.name] = INT16_PLATFORM

# Workaround FP32 Concat Limits
fp32_layers = {"/model.23/Concat_5", "/model.23/Concat_3", "/model.23/Concat_4"}
for op in main_ops:
    if op.name in fp32_layers:
        dispatching_table[op.name] = TargetPlatform.FP32

print("Applying Dispatcher Types...")
for op in graph.operations.values():
    quantizer.quantize_operation(op_name=op.name, platform=dispatching_table[op.name])

Applying Dispatcher Types...


In [13]:
# JE67sJLdH2SP
# ==========================================
# CELL 8: Linear Optimization Pipeline (PTQ + TQT + LUT)
# ==========================================
print("Running Linear Optimization Pipeline (Calibration -> TQT -> LUT)...")
data_cfg = check_det_dataset(QATConfig.DATA_YAML_FILE)
cali_loader = get_calibration_loader(data_cfg)

executor = TorchExecutor(graph=graph)
# The ONNX model was exported with a batch size of 16. The `tracing_operation_meta`
# function needs a dummy input that matches this batch size to correctly analyze the graph.
dummy_input = torch.zeros([16,3, QATConfig.IMG_SZ, QATConfig.IMG_SZ]).to(QATConfig.DEVICE)
executor.tracing_operation_meta(inputs=dummy_input)

pipeline = PFL.Pipeline([
    QuantizeSimplifyPass(),
    QuantizeFusionPass(activation_type=quantizer.activation_fusion_types),
    ParameterQuantizePass(),
    RuntimeCalibrationPass(method=QATConfig.QUANT_CALIB_METHOD),

    # Accuracy Recovery
    TrainedQuantizationThresholdPass(
        steps=QATConfig.TQT_STEPS,
        lr=QATConfig.TQT_LR,
        int_lambda=QATConfig.TQT_INT_LAMBDA,
        block_size=QATConfig.TQT_BLOCK_SIZE,
        collecting_device=QATConfig.TQT_COLLECTING_DEVICE
    ),

    PassiveParameterQuantizePass(clip_visiblity=QuantizationVisibility.EXPORT_WHEN_ACTIVE),
    QuantAlignmentPass(elementwise_alignment=QATConfig.QUANT_ALIGNMENT),

    # Direct LUT Conversion for HW int16 Swish emulation
    EspdlLUTFusionPass(
        target_ops=['Swish'],
        lut_step=QATConfig.INT16_LUT_STEP
    )
])

pipeline.optimize(
    calib_steps=QATConfig.CALIB_STEPS,
    collate_fn=(lambda x: x.type(torch.float).to(QATConfig.DEVICE)),
    graph=graph,
    dataloader=cali_loader,
    executor=executor,
)
print("Pipeline complete.")

Running Linear Optimization Pipeline (Calibration -> TQT -> LUT)...
Using dataset at: /content/Face-Recognation-1/train/images
[WARNING][PPQ][2026-09-23 03:46:57]:  Unexpected input value of operation node_upsample_nearest2d, recieving "None" at its input 1
[WARNING][PPQ][2026-09-23 03:46:57]:  Unexpected input value of operation node_upsample_nearest2d_1, recieving "None" at its input 1
[03:46:57] PPQ Quantize Simplify Pass Running ...         Finished.
[03:46:57] PPQ Quantization Fusion Pass Running ...       Finished.
[03:46:57] PPQ Parameter Quantization Pass Running ...    Finished.
[03:46:57] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 1): 100%|██████████| 16/16 [01:55<00:00,  7.20s/it]


Finished.
[03:48:53] ESP-PPQ TQT Optimization Running ...           
Check following parameters:
Is Scale Trainable:        True
Interested Layers:         []
Num of blocks:             61
Learning Rate:             1e-05
Steps:                     50
Gamma:                     0.0
int_lambda:                0.25

# Block [1 / 61]: [node_conv2d -> node_Split_152]


# Tuning Procedure : 100%|██████████| 50/50 [00:03<00:00, 14.63it/s]


# Tuning Finished  : (1.1566 -> 0.9807) [Block Loss]

# Block [2 / 61]: [node_conv2d_3 -> node_conv2d_4/Swish]


...

# Tuning Finished  : (0.0551 -> 0.0394) [Block Loss]

# Block [60 / 61]: [node_conv2d_118 -> node_conv2d_120]


# Tuning Procedure : 100%|██████████| 50/50 [00:00<00:00, 106.02it/s]


# Tuning Finished  : (0.0771 -> 0.0710) [Block Loss]

# Block [61 / 61]: [node_conv2d_121 -> node_conv2d_125]


# Tuning Procedure : 100%|██████████| 50/50 [00:00<00:00, 72.94it/s]


# Tuning Finished  : (0.4245 -> 0.3984) [Block Loss]

Finished.
[05:31:15] PPQ Passive Parameter Quantization Running ... Finished.
[05:31:15] PPQ Quantization Alignment Pass Running ...    Finished.
[05:31:15] ESPDL LUT Fusion Pass Running ...              Finished.
Pipeline complete.


In [15]:
# krZsCLliH6qv
# ==========================================
# CELL 9: Validation (Check Quantized Graph mAP)
# ==========================================
from trainer import QATTrainer
print("Evaluating Target ESP-DL Emulated mAP...")
dummy_trainer = QATTrainer(graph=graph, model_meta=model_meta, device=QATConfig.DEVICE)
val_mAP = dummy_trainer.eval()
print(f"Final Quantized mAP50-95: {val_mAP:.3f}")

Evaluating Target ESP-DL Emulated mAP...
Ultralytics 8.4.7 🚀 Python-3.12.13 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 722.7±188.1 MB/s, size: 35.3 KB)
val: Scanning /content/Face-Recognation-1/valid/labels.cache... 1042 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1042/1042 132.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 98% ━━━━━━━━━━━╸ 65/66 1.4it/s 36.6s<0.7s


RuntimeError: Op Execution Error: node__unsafe_view(Type: Reshape, Num of Input: 2, Num of Output: 1)

In [ ]:
# HS5GHFAkH_Of
# ==========================================
# CELL 10: Graph Surgery (Pruning and Output Tearing)
# ==========================================
print("Slicing output Concat nodes into 6 discrete tensors...")

# 1. Remove Aux Heads
output_names = list(graph.outputs.keys())
if len(output_names) >= 6:
    for name in output_names[0:3]:
        if name in graph.outputs: graph.outputs.pop(name)
    prune_graph_safely(graph)

# 2. Slice the Concat into Box/Cls
targets = ["one2one_p3", "one2one_p4", "one2one_p5"]
collected_outputs = {}
for target_name in targets:
    if target_name in graph.outputs:
        original_output_var = graph.variables[target_name]
        producer = original_output_var.source_op

        if producer and producer.type == "Concat":
            box_var, cls_var = None, None
            for input_var in producer.inputs:
                dims = input_var.shape
                if dims is not None:
                    if 1 in dims: box_var = input_var
                    elif model_meta['nc'] in dims: cls_var = input_var

            if box_var and cls_var:
                pair_config = [
                    (box_var, f"{target_name}_box"),
                    (cls_var, f"{target_name}_cls"),
                ]
                for var, new_name in pair_config:
                    old_name = var.name
                    if old_name in graph.variables: graph.variables.pop(old_name)
                    var._name = new_name
                    graph.variables[new_name] = var
                    collected_outputs[new_name] = var

                graph.outputs.pop(target_name)
                graph.remove_operation(producer, keep_coherence=False)
                for var in producer.inputs:
                    if producer in var.dest_ops: var.dest_ops.remove(producer)

# 3. Enforce precise output order matching ESPdl C++ expectations
final_output_list = [
    "one2one_p3_box", "one2one_p3_cls",
    "one2one_p4_box", "one2one_p4_cls",
    "one2one_p5_box", "one2one_p5_cls"
]
graph.outputs.clear()
for name in final_output_list:
    if name in collected_outputs:
        graph.outputs[name] = collected_outputs[name]

prune_graph_safely(graph)
print("✅ Salidas del grafo recortadas y estructuradas correctamente.")

In [ ]:
%matplotlib inline
import os
import glob
from notebook_helpers import eval_espdl_model

# 1. Buscar y confirmar la presencia del modelo .espdl en Google Drive
quant_dir = f"{DRIVE_SAVE_PATH}_Quantized"
espdl_matches = glob.glob(os.path.join(quant_dir, "*.espdl"))
if espdl_matches:
    espdl_path = espdl_matches[0]
    print(f"✅ Modelo .espdl encontrado en Drive: {espdl_path}")
else:
    print(f"⚠️ Advertencia: No se encontró ningún archivo .espdl en '{quant_dir}'.")
    print("Asegúrate de haber ejecutado la celda de exportación final.")
    espdl_path = None # Set to None if not found, to avoid errors later

# 2. Configurar rutas de imagen de prueba y carpeta de salida
TEST_IMAGE = '/content/drive/MyDrive/Colab Notebooks/Imagenes de Prueba/33_Running_Running_33_332.jpg'
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/Imagenes de Prueba/Resultado'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Validar el objeto 'graph' en memoria y ejecutar la inferencia emulada
if 'graph' in locals() and graph is not None:
    if espdl_path: # Only proceed if an espdl model was found
        predictions, saved_path = eval_espdl_model(
            test_image_path = TEST_IMAGE,
            graph           = graph,
            target_img_sz   = QATConfig.IMG_SZ,
            data_yaml       = QATConfig.DATA_YAML_FILE,
            platform        = PLATFORM,
            conf_thresh     = 0.25,
            output_dir      = OUTPUT_DIR,
        )
        print(f"\nDetecciones ({len(predictions)}):")
        for p in predictions:
            print(f"  [{p['class_id']:3d}] {p['class']:20s}  conf={p['score']:.3f}  box={[round(v) for v in p['box']]}")
    else:
        print("❌ ERROR: No se encontró un modelo .espdl para la inferencia.")
else:
    print("❌ ERROR: El objeto 'graph' no está disponible en memoria. Ejecuta la celda del pipeline de optimización primero.")

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# wHSGbw7pIKPg
# ==========================================
# CELL 11: Final Export (Auto-Switching Streaming/Standard)
# ==========================================
final_espdl_path = os.path.join(QATConfig.ESPDL_OUTPUT_DIR, f"{QATConfig.MODEL_NAME}_{QATConfig.IMG_SZ}_s8_{PLATFORM}.espdl")

# Using the properly initialized local exporter
exporter = PFL.Exporter(platform=QATConfig.TARGET_PLATFORM)
exporter.export(final_espdl_path, graph=graph, int16_lut_step=QATConfig.INT16_LUT_STEP)
print(f"Deployment Model exported seamlessly to {final_espdl_path}")
print(f"✅ ¡Modelo .espdl exportado exitosamente!: {final_espdl_path}")